## Most common HLA class 2 alleles in the world
Get data for MHC2 from the afnd.tsv file to find the most common MHC2 varieties in the world. 
Use the helper functions from utils.py if it make sense
Reference EDA3.ipynb and EDA2.ipynb since they are doing the same thing but with HLA class 1. 

In [1]:

# Step 1: Load the data and inspect class 2 genes
import pandas as pd
from utils import *

df_raw = pd.read_csv("afnd.tsv", sep="\t")

class2_genes = ['DRB1', 'DQB1', 'DQA1', 'DPB1', 'DPA1']
hla = df_raw[df_raw['group'] == 'hla']
class2_raw = hla[hla['gene'].isin(class2_genes)].copy()

print("=== Class 2 gene row counts ===")
print(class2_raw['gene'].value_counts())

print("\n=== Sample alleles per gene ===")
for gene in class2_genes:
    sample = class2_raw[class2_raw['gene'] == gene]['allele'].unique()[:5]
    print(f"{gene}: {list(sample)}")

print("\n=== Colon count (resolution) in allele names ===")
class2_raw['colon_count'] = class2_raw['allele'].apply(lambda a: str(a).count(':'))
print(class2_raw['colon_count'].value_counts().sort_index())

print("\n=== G-groups present? (asterisk in alleles_over_2n) ===")
g_groups = class2_raw[class2_raw['alleles_over_2n'].astype(str).str.contains(r'\*', regex=True)]
print(f"G-group rows: {len(g_groups)}")


=== Class 2 gene row counts ===
gene
DRB1    36011
DQB1     8596
DPB1     5667
DQA1     2499
DPA1      438
Name: count, dtype: int64

=== Sample alleles per gene ===
DRB1: ['DRB1*01', 'DRB1*03', 'DRB1*04', 'DRB1*07', 'DRB1*08']
DQB1: ['DQB1*02', 'DQB1*03', 'DQB1*04', 'DQB1*05', 'DQB1*06']
DQA1: ['DQA1*01', 'DQA1*02', 'DQA1*03', 'DQA1*04', 'DQA1*05']
DPB1: ['DPB1*01', 'DPB1*02', 'DPB1*03', 'DPB1*04', 'DPB1*05']
DPA1: ['DPA1*01', 'DPA1*02', 'DPA1*03', 'DPA1*04', 'DPA1*01:03']

=== Colon count (resolution) in allele names ===
colon_count
0     6204
1    42124
2     4790
3       93
Name: count, dtype: int64

=== G-groups present? (asterisk in alleles_over_2n) ===
G-group rows: 618


In [ ]:
# Check resolution in the raw dataframe (before any processing)
print("=== Resolution in raw class2 data (class2_raw) ===")
class2_raw['resolution'] = class2_raw['allele'].apply(get_allele_resolution)
print(class2_raw['resolution'].value_counts())

print("\n=== Resolution breakdown per gene ===")
print(class2_raw.groupby(['gene', 'resolution']).size().unstack(fill_value=0))

print("\n=== Compare with class 1 in raw data ===")
class1_raw = hla[hla['gene'].isin(['A', 'B', 'C'])].copy()
class1_raw['resolution'] = class1_raw['allele'].apply(get_allele_resolution)
print(class1_raw['resolution'].value_counts())


=== Resolution in raw class2 data (class2_raw) ===
resolution
4-digit    42124
2-digit     6204
6-digit     4790
8-digit       93
Name: count, dtype: int64

=== Resolution breakdown per gene ===
resolution  2-digit  4-digit  6-digit  8-digit
gene                                          
DPA1             23      211      198        6
DPB1             86     5037      525       19
DQA1            211     2102      177        9
DQB1            864     6855      860       17
DRB1           5020    27919     3030       42

=== Compare with class 1 in raw data ===
resolution
4-digit    69275
2-digit    22010
6-digit     6580
8-digit      624
Name: count, dtype: int64


In [2]:

# Step 2: Test clean_data with class1_only=False
df_class2 = clean_data(df_raw, class1_only=False, verbose=True)
df_class2 = df_class2[df_class2['gene'].isin(class2_genes)]

print("\n=== After clean_data (class2 only) ===")
print(df_class2['gene'].value_counts())
print("\nSample rows:")
print(df_class2.head())


Starting shape: (163803, 7)
After filtering for HLA: (151700, 7)
After removing 2147 G-group rows: (149553, 8)
Final shape after dropping columns: (149553, 5)

=== After clean_data (class2 only) ===
gene
DRB1    35517
DQB1     8546
DPB1     5593
DQA1     2499
DPA1      438
A           0
B           0
C           0
Name: count, dtype: int64

Sample rows:
      gene   allele                      population  alleles_over_2n    n
4517  DPB1  DPB1*01          Argentina Buenos Aires           0.0610  466
4518  DPB1  DPB1*01   Colombia Amazon Region Vaupes           0.0000   46
4519  DPB1  DPB1*01  England North West Mixed pop 2           0.0633  379
4520  DPB1  DPB1*01                France Southeast           0.0550  130
4521  DPB1  DPB1*01                   Myanmar Bamar           0.0217   46


In [3]:
# Check all unique HLA genes in the database
print("All unique genes in the HLA group:")
all_hla_genes = hla['gene'].unique()
print(sorted(all_hla_genes))

# Cross-reference with known HLA class 2 genes
known_class2 = {'DRA', 'DRB1', 'DRB3', 'DRB4', 'DRB5', 'DQA1', 'DQB1', 'DPA1', 'DPB1', 'DMA', 'DMB', 'DOA', 'DOB'}
found_in_db = set(all_hla_genes) & known_class2
missing_from_db = known_class2 - set(all_hla_genes)

print(f"\nKnown class 2 genes found in DB: {sorted(found_in_db)}")
print(f"Known class 2 genes NOT in DB:  {sorted(missing_from_db)}")

# Check if any HLA genes exist that aren't class 1 (A/B/C) or the 5 we picked
class1_genes = {'A', 'B', 'C'}
accounted_for = class1_genes | set(class2_genes)
unaccounted = set(all_hla_genes) - accounted_for
print(f"\nHLA genes not in class1 (A,B,C) and not in our class2 list: {sorted(unaccounted)}")


All unique genes in the HLA group:
['A', 'B', 'C', 'DPA1', 'DPB1', 'DQA1', 'DQB1', 'DRB1']

Known class 2 genes found in DB: ['DPA1', 'DPB1', 'DQA1', 'DQB1', 'DRB1']
Known class 2 genes NOT in DB:  ['DMA', 'DMB', 'DOA', 'DOB', 'DRA', 'DRB3', 'DRB4', 'DRB5']

HLA genes not in class1 (A,B,C) and not in our class2 list: []


In [4]:

# Step 3: Test extract_allele_parts and get_allele_resolution on class 2 alleles
test_alleles = ['DRB1*01:01', 'DQB1*02:01', 'DQA1*01:03', 'DPB1*04:01', 'DPA1*01:03']
print("=== extract_allele_parts on class 2 alleles ===")
for a in test_alleles:
    result = extract_allele_parts(a)
    print(f"{a} -> {result}")

print("\n=== Resolution distribution after clean_data ===")
df_class2 = df_class2[df_class2['gene'].isin(class2_genes)].copy()
df_class2['resolution'] = df_class2['allele'].apply(get_allele_resolution)
print(df_class2['resolution'].value_counts())

print("\n=== All class 2 alleles are 4-digit only — no 6/8-digit collapse needed ===")


=== extract_allele_parts on class 2 alleles ===
DRB1*01:01 -> {'locus': 'DRB1', '2digit': 'DRB1*01', '4digit': 'DRB1*01:01', '6digit': None, '8digit': None, 'resolution': '4-digit'}
DQB1*02:01 -> {'locus': 'DQB1', '2digit': 'DQB1*02', '4digit': 'DQB1*02:01', '6digit': None, '8digit': None, 'resolution': '4-digit'}
DQA1*01:03 -> {'locus': 'DQA1', '2digit': 'DQA1*01', '4digit': 'DQA1*01:03', '6digit': None, '8digit': None, 'resolution': '4-digit'}
DPB1*04:01 -> {'locus': 'DPB1', '2digit': 'DPB1*04', '4digit': 'DPB1*04:01', '6digit': None, '8digit': None, 'resolution': '4-digit'}
DPA1*01:03 -> {'locus': 'DPA1', '2digit': 'DPA1*01', '4digit': 'DPA1*01:03', '6digit': None, '8digit': None, 'resolution': '4-digit'}

=== Resolution distribution after clean_data ===
resolution
4-digit    41851
2-digit     6204
6-digit     4445
8-digit       93
Name: count, dtype: int64

=== All class 2 alleles are 4-digit only — no 6/8-digit collapse needed ===


In [5]:
# Step 4: Diagnose why extract_allele_parts returns None for class 2 alleles
import re

# Check the raw allele string values in the cleaned dataframe
sample_alleles = df_class2['allele'].head(5).tolist()
print("Raw allele values from df_class2:", sample_alleles)
print("Types:", [type(a) for a in sample_alleles])

# Try matching manually
for a in sample_alleles:
    a_str = str(a)
    match = re.match(r'([A-Z]+)\*(\d+)(?::(\d+))?(?::(\d+))?(?::(\d+))?', a_str)
    print(f"  repr: {repr(a_str)} -> match: {match}")


Raw allele values from df_class2: ['DPA1*01:03', 'DPA1*01:03', 'DPA1*01:03', 'DPA1*01:03', 'DPA1*01:03']
Types: [<class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>, <class 'str'>]
  repr: 'DPA1*01:03' -> match: None
  repr: 'DPA1*01:03' -> match: None
  repr: 'DPA1*01:03' -> match: None
  repr: 'DPA1*01:03' -> match: None
  repr: 'DPA1*01:03' -> match: None


In [6]:
# Step 5: Reload utils with the fix and re-test extract_allele_parts
import importlib, utils
importlib.reload(utils)
from utils import *

test_alleles = ['DRB1*01:01', 'DQB1*02:01', 'DQA1*01:03', 'DPB1*04:01', 'DPA1*01:03']
print("=== extract_allele_parts after regex fix ===")
for a in test_alleles:
    result = extract_allele_parts(a)
    print(f"  {a} -> {result}")


=== extract_allele_parts after regex fix ===
  DRB1*01:01 -> {'locus': 'DRB1', '2digit': 'DRB1*01', '4digit': 'DRB1*01:01', '6digit': None, '8digit': None, 'resolution': '4-digit'}
  DQB1*02:01 -> {'locus': 'DQB1', '2digit': 'DQB1*02', '4digit': 'DQB1*02:01', '6digit': None, '8digit': None, 'resolution': '4-digit'}
  DQA1*01:03 -> {'locus': 'DQA1', '2digit': 'DQA1*01', '4digit': 'DQA1*01:03', '6digit': None, '8digit': None, 'resolution': '4-digit'}
  DPB1*04:01 -> {'locus': 'DPB1', '2digit': 'DPB1*04', '4digit': 'DPB1*04:01', '6digit': None, '8digit': None, 'resolution': '4-digit'}
  DPA1*01:03 -> {'locus': 'DPA1', '2digit': 'DPA1*01', '4digit': 'DPA1*01:03', '6digit': None, '8digit': None, 'resolution': '4-digit'}


In [7]:
# Step 6: Run the full pipeline on class 2 data (mirrors test_pipeline.ipynb)
import importlib, utils
importlib.reload(utils)
from utils import *
import os

# 1. Clean: filter HLA group, remove G-groups, keep all genes (not class1_only)
df_clean = clean_data(df_raw, class1_only=False, remove_g_groups=True, verbose=False)

# 2. Isolate class 2 genes
df_clean = df_clean[df_clean['gene'].isin(class2_genes)].copy()
print(f"After isolating class 2 genes: {df_clean.shape}")

# 3. Collapse to 4-digit resolution
if os.path.exists("4_digit_class2.csv"):
    df_4digit = pd.read_csv("4_digit_class2.csv")
else:
    df_4digit = collapse_to_4digit(df_clean, remove_inconsistent_studies=True, max_2digit_diff=0.005, verbose=True)
    df_4digit.to_csv("4_digit_class2.csv", index=False)
print(f"After collapse_to_4digit: {df_4digit.shape}")

# 4. Clean and normalize (min_sample_size=1000, normalize frequencies)
df_final = clean_and_normalize(df_4digit, freq_sum_threshold=0.1, min_sample_size=1000, normalize=True, cleaning_method='population', verbose=True)


After isolating class 2 genes: (52593, 5)
Starting collapse_to_4digit
Input shape: (52593, 5)
Input studies: 1077

Resolution distribution before collapse:
{'4-digit': 41851, '2-digit': 6204, '6-digit': 4445, '8-digit': 93}

--- Step 1: Collapse 8-digit → 6-digit alleles ---


Collapsing 8-digit to 6-digit: 100%|██████████| 1077/1077 [00:03<00:00, 358.52it/s]


Collapsed 8-digit to 6-digit: (52593, 6) -> (52577, 6)
  Updates: 13
  Created: 77

--- Step 2: Collapse 6-digit → 4-digit alleles ---


Collapsing 6-digit to 4-digit: 100%|██████████| 1077/1077 [00:26<00:00, 40.93it/s]


Collapsed 6-digit to 4-digit: (52577, 6) -> (51558, 6)
  Updates: 235
  Created: 3503

--- Step 3: Remove studies where 2-digit parent freq > sum of 4-digit children ---
  Using max_2digit_diff threshold: 0.005


Finding 2-digit inconsistencies: 100%|██████████| 1077/1077 [00:15<00:00, 69.33it/s] 


Removed 109 studies with total_diff > 0.005
  Shape: (51558, 6) -> (47794, 6)
  Studies: 1077 -> 968

--- Step 4: Remove 2-digit entries ---
Removed 4485 2-digit entries
  Shape: 47794 -> 43309

collapse_to_4digit complete!
Final shape: (43309, 6)
Final studies: 747
Resolution distribution: {'4-digit': 43309}
After collapse_to_4digit: (43309, 6)
Starting clean_and_normalize
Input shape: (43309, 6)
Input studies: 747

--- Step 1: Validate and remove studies with invalid frequency sums ---
  Valid range: [1-0.1, 1+0.1] = [0.9, 1.1]
  Method: population (remove entire population if any gene is invalid)
Removed 209 populations with at least one invalid gene combination
  Shape: (43309, 6) -> (33803, 6)
  Studies: 747 -> 538

--- Step 2: Remove entries where freq is 0 ---
Removed 11365 entries with 0 frequency
  Shape: 33803 -> 22438

--- Step 3: Normalize frequencies to 1 in each (population, gene) combination ---
Normalized frequencies for 1103 (population, gene) combinations
  Frequency 

In [8]:
# Step 7: Verify pipeline output and find most common class 2 alleles
print(f"Final dataset: {df_final.shape[0]} rows, {df_final['population'].nunique()} studies")
print(f"Genes: {sorted(df_final['gene'].unique())}")
print(f"Resolution: {df_final['resolution'].unique()}")

freq_sums = df_final.groupby(['population', 'gene'])['alleles_over_2n'].sum()
print(f"Frequency sum range: [{freq_sums.min():.6f}, {freq_sums.max():.6f}]")

# Most common allele per gene: average frequency across all studies
print("\n=== Top 5 most common alleles per gene (by mean frequency) ===")
for gene in sorted(df_final['gene'].unique()):
    top = (df_final[df_final['gene'] == gene]
           .groupby('allele')['alleles_over_2n']
           .mean()
           .sort_values(ascending=False)
           .head(5))
    print(f"\n{gene}:")
    for allele, freq in top.items():
        print(f"  {allele}: {freq:.4f} ({freq*100:.2f}%)")


Final dataset: 6656 rows, 86 studies
Genes: ['DPA1', 'DPB1', 'DQA1', 'DQB1', 'DRB1']
Resolution: ['4-digit']
Frequency sum range: [1.000000, 1.000000]

=== Top 5 most common alleles per gene (by mean frequency) ===

DPA1:
  DPA1*01:03: 0.4921 (49.21%)
  DPA1*02:02: 0.3868 (38.68%)
  DPA1*02:01: 0.0923 (9.23%)
  DPA1*04:01: 0.0269 (2.69%)
  DPA1*02:07: 0.0103 (1.03%)

DPB1:
  DPB1*04:01: 0.2434 (24.34%)
  DPB1*05:01: 0.1574 (15.74%)
  DPB1*02:01: 0.1542 (15.42%)
  DPB1*04:02: 0.1295 (12.95%)
  DPB1*03:01: 0.1197 (11.97%)

DQA1:
  DQA1*05:03: 0.2113 (21.13%)
  DQA1*01:02: 0.1290 (12.90%)
  DQA1*03:02: 0.1117 (11.17%)
  DQA1*06:01: 0.0972 (9.72%)
  DQA1*01:03: 0.0874 (8.74%)

DQB1:
  DQB1*03:01: 0.2239 (22.39%)
  DQB1*03:22: 0.1971 (19.71%)
  DQB1*02:01: 0.1357 (13.57%)
  DQB1*03:02: 0.0997 (9.97%)
  DQB1*05:01: 0.0926 (9.26%)

DRB1:
  DRB1*07:01: 0.1081 (10.81%)
  DRB1*03:01: 0.0822 (8.22%)
  DRB1*15:01: 0.0773 (7.73%)
  DRB1*11:04: 0.0642 (6.42%)
  DRB1*11:01: 0.0513 (5.13%)


In [9]:
df_final.to_csv("cleaned_data_normal_class2.csv")
print(f"Saved to cleaned_data_normal_class2.csv ({len(df_final)} rows)")


Saved to cleaned_data_normal_class2.csv (6656 rows)


In [10]:
# Gene distribution in the final dataset
gene_counts = df_final['gene'].value_counts()
gene_pct = df_final['gene'].value_counts(normalize=True) * 100

summary = pd.DataFrame({'rows': gene_counts, 'pct_rows': gene_pct.round(1),
                        'num_studies': df_final.groupby('gene')['population'].nunique(),
                        'num_alleles': df_final.groupby('gene')['allele'].nunique()})
print(summary.to_string())


      rows  pct_rows  num_studies  num_alleles
gene                                          
DPA1    28       0.4            4           11
DPB1   380       5.7            8          204
DQA1    63       0.9            4           18
DQB1   723      10.9           28          203
DRB1  5462      82.1           83          552


## Summary
- Update raw afnd.tsv to include the different resolutions for class 2 HLAs. 
- Only 5 HLA class 2 genes present in the database: ['DPA1', 'DPB1', 'DQA1', 'DQB1', 'DRB1']
- Can use the same pipeline with some small modifications
- Output data in cleaned_data_normal_class2.csv. 